<!-- STATUS_BLOCK_v1 -->
# statistical_methods.ipynb — STATUS

**Pipeline position:** Pipeline [5] — reading guide

**Purpose.**  Pedagogical reference for interpreting every test and threshold in `5g_foraging_effect_model.ipynb`: MWU, rank-biserial r, bootstrap CI, Spearman, BH-FDR, mixed-effects coefficients, the 5-check decision rule, composite FII.

**Inputs:**  none (illustrative numbers use current data state)
**Outputs:** explanatory text only

### WORKS
- 10 sections covering every artefact the model produces
- Worked examples populated from real data values
- Cheat sheet for reading a verdict report

### PENDING
- Update r-value examples after the indicator swap (median_ifi_s → mean_handling_time_s, vertical_deviation → n_distinct_flowers)

## Pipeline flow (canonical)

```
data/flight_data/<date>_system_<sys>/                  ← raw PATS-C output
       │
       ▼
[1] flower_visit_pipeline.ipynb                        slow; run when raw data changes
       └── data/multi_day/flower_visit_summary.csv

[2] multi_day_pipeline.ipynb                           always run after raw data updates
       ├── data/multi_day/daily_summary.csv
       ├── data/multi_day/per_track_indicators.csv
       └── data/multi_day/indicators_daily.csv         ← the file the model consumes

       (uses outputs of [1] + dBm + data transfer)
       │
       ▼
[3] validation.ipynb                                   sensor-integrity QC, run anytime
[3] indicator_validation.ipynb                         baseline-only QC of the 6 indicators
       │
       ▼
[4] exposure_analysis.ipynb                            figures + exploratory plots
[4] 5g_foraging_effect_model.ipynb                     pre-registered verdict, FINAL output
       │
       ▼
[5] statistical_methods.ipynb                          reading guide for [4]; not data-dependent
       │
       ▼
   paper / report
```

**Used in final report:**
- `5g_foraging_effect_model.ipynb` (verdict, mixed-effects model, composite FII)
- `exposure_analysis.ipynb` (figures)
- `validation.ipynb` (Methods)
- `indicator_validation.ipynb` (Methods)
- `statistical_methods.ipynb` (reference)

---


# Statistical methods reference

This notebook is a reading guide for the outputs of `5g_foraging_effect_model.ipynb`.

For each test we use it tells you:

1. **What the test answers** (the question)
2. **How to read the output** (the threshold table)
3. **What to write in your paper** (the report-language convention)
4. **A worked example** using the actual data we have

The model uses *six* statistical artefacts:

| § | artefact | answers |
|---|---|---|
| 1 | Mann-Whitney U (MWU) | is the distribution of indicator X in ON different from OFF? |
| 2 | rank-biserial r | how *big* is that ON-vs-OFF difference, in dimensionless units? |
| 3 | Bootstrap 95 % CI on r | what's the *uncertainty* on the effect size? |
| 4 | Spearman ρ | does indicator X track a continuous variable like dBm or temperature? |
| 5 | Benjamini-Hochberg q | adjusts p-values for multiple comparisons (one indicator × many tests) |
| 6 | Mixed-effects model coef | what's the ON-vs-OFF difference *after* accounting for system and day random effects? |

There is also the **5-check decision rule** and the **composite FII index** —
documented in sections 7 and 8 — which are how we combine the six
individual tests into a single PASS / SUGGESTIVE / FAIL verdict for the
hypothesis "5G impairs bumblebee foraging".


## 1. Mann-Whitney U test

**Question.** *Are values in group A drawn from a distribution that
produces systematically larger (or smaller) values than the distribution
behind group B?*

We use MWU (not the t-test) because:

- n is small per group (typically 6-9 day-system rows per condition)
- the data are not normally distributed (count-like for `n_exit_count`,
  positively skewed for `mean_handling_time_s`, bounded for `re_ratio`)
- MWU is robust to outlier day-system rows
- "two-sided" because we don't pre-specify the direction the indicator
  should move (although for the *composite* FII we do pre-specify
  direction — see § 8)

**How to read the output:**

| p-value | what it means | report as |
|---|---|---|
| p < 0.001 | very strong evidence the distributions differ | "significant (p < .001)" |
| p < 0.01 | strong evidence | "significant (p < .01)" |
| p < 0.05 | conventional significance threshold | "significant (p = X)" |
| 0.05 ≤ p < 0.10 | suggestive but not significant | "marginal (p = X)" or "trend" |
| p ≥ 0.10 | no evidence of a difference | "no significant difference (p = X)" |

**Critical caveat for this dataset.** With n = 6 vs 6 the MWU has a
theoretical floor of p ≈ 0.002 (when the two groups don't overlap *at
all*). Above that floor you need a very large shift to even reach
p < 0.05. **Don't expect significance at n = 6.** That's why we lean on
effect size (§ 2) and replication (§ 7.3) instead.


In [ ]:
# Worked example - MWU on n_v3 (sys 900, OFF vs ON) from current data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
daily = pd.read_csv("data/multi_day/daily_summary.csv", parse_dates=["date"])
CYCLE = pd.Timestamp("2026-04-23")
def lbl(d):
    if d < CYCLE: return "BASELINE"
    return "ON" if ((d-CYCLE).days // 3) % 2 == 0 else "OFF"
daily["condition"] = daily["date"].apply(lbl)

off = daily[(daily.system_id == 900) & (daily.condition == "OFF")]["n_v3"].values
on  = daily[(daily.system_id == 900) & (daily.condition == "ON")]["n_v3"].values

# inline MWU (no scipy dependency)
def mwu(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    n1, n2 = len(a), len(b)
    combined = np.concatenate([a, b])
    ranks = pd.Series(combined).rank().values
    U1 = ranks[:n1].sum() - n1*(n1+1)/2
    U2 = n1*n2 - U1
    U = min(U1, U2)
    mu = n1*n2/2
    sigma = np.sqrt(n1*n2*(n1+n2+1)/12)
    from math import erf, sqrt
    z = (U - mu) / sigma if sigma else 0
    p = 2*(1 - 0.5*(1 + erf(abs(z)/sqrt(2))))
    return U, p

U, p = mwu(on, off)
print(f"n_v3 ON  (sys 900): median = {np.median(on):.0f}   n = {len(on)}   values = {sorted(on)}")
print(f"n_v3 OFF (sys 900): median = {np.median(off):.0f}   n = {len(off)}   values = {sorted(off)}")
print(f"\nMann-Whitney U = {U:.0f}")
print(f"two-sided p     = {p:.4f}")
print(f"\nInterpretation: p > 0.05 -> not significant at conventional threshold.")
print(f"But median ON < median OFF, so the direction is consistent with H1.")


## 2. Rank-biserial correlation r (effect size)

**Question.** *If I pick one random day from ON and one from OFF, how
often does the ON value exceed the OFF value?*

For two samples of size n₁ (group A) and n₂ (group B):

```
r = 2U₁ / (n₁ · n₂) − 1
```

where U₁ is the Mann-Whitney U statistic for group A.

**How to read it:**

| r value | what it means |
|---|---|
| r = +1.00 | every ON day exceeds every OFF day (no overlap, ON ≫ OFF) |
| r = +0.50 | strong shift — ON usually exceeds OFF |
| r = +0.30 | moderate shift |
| r = +0.10 | small shift |
| r = 0     | groups are interchangeable (no effect) |
| r = -0.30 | moderate shift, OFF > ON |
| r = -1.00 | every OFF day exceeds every ON day |

**Thresholds we apply** (matched to standard psychology / ecology
conventions, Cohen 1988 mapped to rank-biserial):

| |r| range | Cohen-equivalent | interpretation |
|---|---|---|
| 0.00-0.10 | negligible | no effect |
| 0.10-0.30 | small | weak effect, unlikely to be detected at n = 6 |
| 0.30-0.50 | medium | reportable effect if replicated |
| 0.50-0.80 | large | strong effect |
| 0.80-1.00 | very large | near-deterministic shift |

**Reporting convention.** Cite r with its 95 % CI from bootstrap (§ 3).
For example: "n_v3 was lower under ON than OFF (rank-biserial r = -0.42,
95 % CI [-0.81, +0.04], p = 0.07)."


In [ ]:
# Worked example: r for n_v3 (sys 900, ON vs OFF)
def rank_biserial(on_vals, off_vals):
    n1, n2 = len(on_vals), len(off_vals)
    combined = np.concatenate([on_vals, off_vals])
    ranks = pd.Series(combined).rank().values
    U_on = ranks[:n1].sum() - n1*(n1+1)/2
    return 2*U_on/(n1*n2) - 1

r = rank_biserial(on, off)
print(f"n_v3 (sys 900, ON vs OFF): rank-biserial r = {r:+.3f}")
print(f"Interpretation: r between -0.30 and -0.50 -> moderate negative effect (ON < OFF).")

# Show all 6 indicator-level effect sizes for the current data
from pathlib import Path
fv = pd.read_csv("data/multi_day/flower_visit_summary.csv", parse_dates=["date"])
fv["condition"] = fv["date"].apply(lbl)

# build the indicator table
ind = daily[["date","system_id","condition","n_v3","re_ratio_v3"]].copy()
ind["neg_exit_count"] = -ind["n_v3"]
ind["neg_re_ratio"]   = -ind["re_ratio_v3"]
ind = ind.merge(fv[["date","system_id","mean_handling_time_s","n_distinct_flowers"]],
                on=["date","system_id"], how="left")

print()
print(f"{'indicator':<28s}  {'r':>7s}  {'|r| size':>10s}  {'direction':<24s}")
for ind_name in ["neg_exit_count","neg_re_ratio","mean_handling_time_s","n_distinct_flowers"]:
    on_vals  = ind.loc[(ind.system_id==900) & (ind.condition=="ON"),  ind_name].dropna().values
    off_vals = ind.loc[(ind.system_id==900) & (ind.condition=="OFF"), ind_name].dropna().values
    if len(on_vals) < 3 or len(off_vals) < 3: continue
    r = rank_biserial(on_vals, off_vals)
    size = ("negligible" if abs(r)<0.10 else "small" if abs(r)<0.30 else
            "medium" if abs(r)<0.50 else "large" if abs(r)<0.80 else "very large")
    dir = "ON > OFF (H1)" if r > 0 else "ON < OFF (against H1)"
    print(f"{ind_name:<28s}  {r:+7.3f}  {size:>10s}  {dir:<24s}")


## 3. Bootstrap 95 % confidence interval on r

**Question.** *If we re-ran this experiment, what's the plausible range
of effect sizes we'd see, given our data?*

We resample (with replacement) the ON and OFF days B = 5000 times,
recompute r for each resample, and take the 2.5th and 97.5th percentiles
as the 95 % CI.

**How to read it:**

| CI shape | interpretation |
|---|---|
| CI entirely positive (`+0.10, +0.62`) | confident the effect is positive |
| CI entirely negative (`-0.55, -0.10`) | confident the effect is negative |
| CI crosses zero (`-0.08, +0.55`) | direction is uncertain even though point estimate ≠ 0 |
| CI is wide (`-0.95, +0.95`) | tiny sample — almost no information |
| CI is narrow (`+0.25, +0.45`) | well-powered estimate |

**Reporting convention.** Always report r with its CI. The CI is *more
informative* than the p-value at small n, because it tells you both
*magnitude* and *uncertainty*.

**Rule of thumb.** With n = 6 vs 6 our CIs are wide (often half the
[-1, +1] range). Treat individual r values as ordinal evidence; only
trust them when they replicate across cycles (§ 7.3) or systems (§ 7.1).


In [ ]:
def bootstrap_r_ci(on_vals, off_vals, B=5000, seed=42):
    rng = np.random.default_rng(seed)
    on_vals = np.asarray(on_vals); off_vals = np.asarray(off_vals)
    rs = np.empty(B)
    for b in range(B):
        a = rng.choice(on_vals,  size=len(on_vals),  replace=True)
        o = rng.choice(off_vals, size=len(off_vals), replace=True)
        rs[b] = rank_biserial(a, o)
    return np.percentile(rs, [2.5, 97.5])

# Run on the 4 main indicators
print(f"{'indicator':<28s}  {'r':>7s}  {'95% CI':>20s}  {'CI crosses 0?':<14s}")
for ind_name in ["neg_exit_count","neg_re_ratio","mean_handling_time_s","n_distinct_flowers"]:
    on_vals  = ind.loc[(ind.system_id==900) & (ind.condition=="ON"),  ind_name].dropna().values
    off_vals = ind.loc[(ind.system_id==900) & (ind.condition=="OFF"), ind_name].dropna().values
    if len(on_vals) < 3 or len(off_vals) < 3: continue
    r = rank_biserial(on_vals, off_vals)
    lo, hi = bootstrap_r_ci(on_vals, off_vals)
    crosses = "YES" if lo < 0 < hi else "no"
    print(f"{ind_name:<28s}  {r:+7.3f}  [{lo:+5.2f}, {hi:+5.2f}]  {crosses:<14s}")


## 4. Spearman rank correlation ρ

**Question.** *Is indicator X monotonically related to a continuous
covariate Z?*

Used for the dose-response check (does the indicator move with dBm
exposure level?) and the weather check (does it just track temperature?).

**How to read it:**

| ρ | interpretation |
|---|---|
| ρ > +0.7 | strong positive monotonic relationship |
| +0.4 < ρ < +0.7 | moderate positive |
| -0.4 < ρ < +0.4 | weak / negligible |
| -0.7 < ρ < -0.4 | moderate negative |
| ρ < -0.7 | strong negative |

We use Spearman (rank-based) rather than Pearson (linear) so a single
outlier day-system row doesn't dominate the result.

**For dose-response:** if `Spearman(indicator, dBm) > +0.4 and p < 0.05`,
the indicator scales with exposure intensity. That's stronger evidence
than ON-vs-OFF alone.


## 5. Benjamini-Hochberg FDR correction

**Question.** *We test 6 indicators × 2 systems = 12 hypotheses. How do
we keep the family-wise false-positive rate under control without making
each test ridiculously conservative?*

BH controls the **false discovery rate (FDR)** — the expected proportion
of "significant" findings that are actually false positives — instead of
the family-wise error rate. It's the right correction when you have many
related tests and care about minimising spurious findings without
zero-tolerance.

**Procedure.** Sort raw p-values smallest to largest. The i-th smallest
gets compared against `(i / m) · α` where m is the number of tests.
The largest p that beats its threshold defines the cutoff; everything
smaller is "significant after BH at α".

**How to read the output:**

A `p_BH < 0.05` means the indicator is significant *after* multiple-test
correction. With m = 12 and uncorrected p = 0.04, the BH-adjusted
`p_BH ≈ 0.24` typically — so corrected significance is much harder to
reach than raw significance.

**Why we use this and not Bonferroni:** Bonferroni would divide α by m
(0.05/12 = 0.0042) which our n = 6 tests cannot achieve. BH is more
permissive because it controls FDR, not FWER.


## 6. Mixed-effects model

**Question.** *After accounting for system-specific baselines and the
particular days we observed, what's the average ON-vs-OFF effect on the
composite FII?*

We fit `FII ~ condition + (1 | system_id)` with `statsmodels.MixedLM`,
where:

- `condition` is the fixed effect we care about (ON = 1, OFF = 0)
- `(1 | system_id)` is a random intercept letting each hive system have
  its own baseline FII

**How to read the output:**

```
                    Coef.    Std.Err.    z     P>|z|
Intercept           -0.05     0.18    -0.28    0.78
condition[T.ON]     +0.41     0.21    +1.95    0.05    <-- the effect we want
```

| field | what it means |
|---|---|
| Coef. (condition[T.ON]) | average FII shift from OFF to ON, in z-units |
| Std.Err. | uncertainty on that shift |
| P>|z| | two-sided test that the shift is zero |
| (sigma_system) | how much the systems differ in baseline FII |

**Reporting convention.** Cite the coefficient and its standard error:
"Composite FII was higher under ON than OFF (mixed-effects β = +0.41,
SE = 0.21, z = 1.95, p = 0.05)."

**Why mixed-effects instead of two-sample t-test:** because system 900
and 939 have different baseline FII values, a pooled t-test would
underestimate the effect by treating system-baseline differences as
within-condition noise. The random intercept absorbs that nuisance
variance.


## 7. The 5-check decision rule

A single p-value at n = 6 is too noisy to support a paper's conclusion.
Instead the model converts six raw tests into five *pre-registered
binary checks*. A check is PASS or FAIL — no in-between. The final
verdict counts how many checks pass.

### 7.1 Check 1 — Cross-system agreement (≥ 4 / 6 indicators)

For each indicator, compute r on sys 900 ON vs OFF and r on sys 939 ON
vs OFF. If both r have the same sign, that indicator "agrees".

**PASS:** at least 4 of 6 indicators agree across systems.

Sign agreement is robust at small n because it asks only for *direction*,
not magnitude. Two independent hive systems agreeing on direction for
4/6 indicators is unlikely by chance (binomial p = 0.34, but the test is
qualitative not quantitative).

### 7.2 Check 2 — Pooled effect size (≥ 3 / 6 with |r| > 0.3)

Pool all day-system rows (n = 12 ON vs 12 OFF), compute r per indicator.
If at least 3/6 indicators have |r| > 0.3, the test passes.

**PASS:** the indicator suite shows medium-or-larger effects on multiple
fronts.

### 7.3 Check 3 — Cross-cycle replication (≥ 4 / 6 indicators)

For each indicator, compute r separately on cycle 0 (first ON block vs
first OFF block) and cycle 1 (second ON vs second OFF). If r has the
same sign in both cycles, the indicator "replicates".

**PASS:** at least 4/6 replicate across cycles.

Replication is the strongest robustness check available at n = 6
because it's *independent*: cycle 1 is a fresh experiment that doesn't
share days with cycle 0.

### 7.4 Check 4 — Weather robustness (≥ 3 / 6 retain |r| > 0.3)

For each indicator, residualise on (T_mean_day, wind_mean_day) via OLS,
then recompute r on the residuals. If the indicator's |r| > 0.3 survives
the regression, it's not just tracking weather.

**PASS:** at least 3/6 indicators retain |r| > 0.3 after weather
adjustment.

### 7.5 Check 5 — Drift-clean (D3 OFF ≈ D1 ON, p > 0.05)

Day-3 of an OFF block (just before transmitter turns on) should look
the same as Day-1 of an ON block (transmitter just turned on, before
any acute effects). If these differ significantly, something is
drifting (acclimatisation, time-of-day artefacts, etc.) and the
ON-vs-OFF comparison is contaminated.

**PASS:** MWU on (D3 OFF) vs (D1 ON) for the composite FII gives
p > 0.05.

### Final verdict

| score | verdict | meaning |
|---|---|---|
| 5/5 | **DETECTED** | strong directional signal, replicated, robust |
| 3-4/5 | **SUGGESTIVE** | some evidence, not all checks passed |
| 1-2/5 | **WEAK/UNDERPOWERED** | dataset cannot distinguish |
| 0/5 | **NOT DETECTED** | no consistent evidence |


## 8. Composite Foraging Impairment Index (FII)

**Why a composite.** Each indicator alone is underpowered at n = 6. But
the 6 indicators are *sign-aligned* (positive = 5G impairs) so averaging
their z-scores per day-system row gives a single number that combines
their information.

**Formula.** Per day-system row:

```
z(indicator_k) = (value − within-system mean) / within-system SD
FII = mean(z(indicator_1), z(indicator_2), ..., z(indicator_6))
```

The within-system normalisation removes the systematic difference
between hive 900 and hive 939 baselines.

**How to read FII:**

| FII | interpretation |
|---|---|
| FII = 0 | matches the within-system baseline |
| FII > 0 | impairment direction (consistent with H1) |
| FII > +0.5 | strong impairment direction |
| FII < 0 | enhancement direction (against H1) |

**The headline test.** MWU comparing pooled ON FII vs pooled OFF FII,
plus bootstrap CI on r. If r > +0.3 with CI not crossing zero, that's
the strongest single piece of evidence for the H1 hypothesis.


## 9. Cheat sheet — reading a verdict report

When `5g_foraging_effect_model.ipynb` prints its final verdict, here's
what each line means:

```
Pre-registered decision rule
[PASS]  1. Cross-system agreement (>=4/6)             <- both hives agree directionally
[PASS]  2. Pooled effect size (>=3/6 with |r|>0.3)    <- meaningful magnitude
[fail]  3. Cross-cycle replication (>=4/6)            <- second cycle didn't reproduce
[fail]  4. Weather-robust (>=3/6 retain |r|>0.3)      <- weather confound
[PASS]  5. FII drift-clean (D3OFF~D1ON, p>0.05)       <- no day-1 vs day-3 contamination

Score: 3/5 checks passed
VERDICT: SUGGESTIVE -- some evidence, not all checks passed

Median FII  ON: +0.20
Median FII OFF: -0.11
rank-biserial r = +0.42  (95% CI -0.04, +0.81)  MWU p = 0.083
```

**Plain-English translation.**

> "There is suggestive evidence that 5G exposure shifts the colony's
> foraging behaviour in the direction predicted by H1. Both hive systems
> agree on the direction across 5 of 6 indicators, and the pooled
> composite index sits +0.42 effect-size units higher under exposure
> (95 % CI [-0.04, +0.81], MWU p = 0.083). However, the effect is not
> reproduced across both experimental cycles at the same magnitude, and
> the systems differed in their typical exposure to wind on
> transmitter-ON days, so the weather-residualised effect sizes shrank
> below our pre-registered threshold for 3 of the 6 indicators. We
> therefore classify the result as suggestive rather than detected."

**When to upgrade to DETECTED:**

- Get more cycles. Each additional 3-day ON/OFF pair roughly doubles the
  power for Check 3.
- Get dBm data. A continuous covariate (Section 4) tests directly
  whether the indicator moves with exposure intensity, much more
  powerful than the binary ON/OFF test.
- Pre-register the indicators *before* peeking at the data. Anything
  decided after data inspection is exploratory.

**When to downgrade to UNDERPOWERED:**

- If both Check 1 and Check 2 fail, the effect (if it exists) is too
  small for this dataset size. Don't keep tweaking thresholds to find
  significance — write up the null and note that the upper CI bound is
  what you've ruled out.


## 10. Glossary of terms

| term | definition |
|---|---|
| **MWU** | Mann-Whitney U, nonparametric test that asks if two distributions are shifted relative to each other |
| **rank-biserial r** | effect size derived from MWU, ranges [-1, +1] |
| **bootstrap CI** | confidence interval estimated by resampling the data with replacement |
| **Spearman ρ** | rank-based correlation coefficient |
| **BH / FDR** | Benjamini-Hochberg correction; controls expected fraction of false positives |
| **mixed-effects model** | regression that allows some intercepts/slopes to vary across grouping (here: per hive system) |
| **random intercept** | letting each group (hive system) have its own baseline value of the response |
| **FII** | Foraging Impairment Index — z-score-averaged composite of the 6 indicators |
| **pre-registered** | hypothesis, test, and decision rule fixed before looking at the data |
| **rank-biserial r vs Cohen's d** | both are effect sizes; r ≈ d / sqrt(d² + 4) for typical sample sizes |
| **CV** (coefficient of variation) | SD divided by mean; dimensionless noise measure |
| **drift-clean** | comparing two adjacent days (last OFF vs first ON) to check for slow-changing artefacts |
| **n** | sample size; in our setup, the number of day-system rows in a group |
